# Milestone 4: Two-Stage Cascade Inference, Calibration Persistence & Clinical Routing

This runnable walkthrough demonstrates the end-to-end Milestone 4 inference and deployment workflow in **MRITumorDetection-CNN**:
1. **Calibrated Stage 1 & Stage 2 Cascade (`CascadePredictor`)**: Constructing calibrated `SimpleCNN` (Stage 1 binary screen) and `TimmClassifier("efficientnet_b0")` (Stage 2 3-class subtyper, `pretrained=False`) and routing synthetic MRI slices across (a) healthy clearance (`no_tumor`), (b) confident tumor subtyping, and (c) confidence abstention (`ABSTAIN_LABEL`).
2. **Selective Grad-CAM & Probability Diagnostics**: Generating `Grad-CAM` overlays (`return_gradcam=True`) for detected tumors and plotting `[Preprocessed Crop | Grad-CAM Overlay | Stage 2 Calibrated Probabilities]`.
3. **Automatic Calibration Persistence (`CascadePredictor.from_configs`)**: Writing `best_model.pt`, `calibration.json`, and temporary YAML configs in a scratch directory and verifying automatic override loading of `stage1_threshold`, `stage2_temperature`, and `stage2_uncertainty_threshold`.
4. **Batched Inference & `4x5` Clinical Routing Matrix**: Running `predict_batch_tensors` over a synthetic test cohort and evaluating `compute_cascade_metrics` + `plot_confusion_matrix`.

## 1. Imports, Reproducibility & Scratch Directory Setup

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

# Ensure src/ is importable if the notebook is run without editable install
repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import cv2
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import yaml

from mri_tumor.evaluation import compute_cascade_metrics, plot_confusion_matrix
from mri_tumor.inference import CascadePredictor, CascadeResult
from mri_tumor.models import SimpleCNN, TimmClassifier
from mri_tumor.training import (
    ABSTAIN_LABEL,
    TemperatureScaler,
    find_sensitivity_threshold,
    set_seed,
)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scratch_ctx = tempfile.TemporaryDirectory(prefix="mri_milestone4_")
scratch_dir = Path(scratch_ctx.name)

def show_saved_plot(path: Path, figsize: tuple[float, float] = (6.8, 5.2)) -> None:
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

print(f"PyTorch: {torch.__version__} | Device: {device} | Scratch dir: {scratch_dir}")

## 2. Synthetic MRI Cohort & Calibrated Stage 1 / Stage 2 Models

We generate synthetic `256x256` brain MRI slices (with dark outer scanner borders so `CascadePredictor.preprocess_array` exercises `crop_brain_contour`, `[0.5, 99.5]` percentile normalization, and `224x224` resizing):
- `no_tumor`: normal brain parenchyma + dark ventricles.
- `glioma`: hyperintense irregular mass in the upper-left hemisphere.
- `meningioma`: bright dural-based mass on the right lateral convexity.
- `pituitary`: compact bright adenoma in the inferior central sellar region.
- `ambiguous`: borderline dual-focus/low-contrast lesion designed to pass Stage 1 screening while triggering Stage 2 confidence abstention (`ABSTAIN_LABEL`).

In [ ]:
def make_synthetic_mri(kind: str, seed_idx: int, raw_size: int = 256) -> np.ndarray:
    """Generates a 2D float32 synthetic MRI slice with dark scanner margins."""
    rng = np.random.default_rng(1000 + seed_idx)
    img = np.zeros((raw_size, raw_size), dtype=np.float32)

    cx = raw_size // 2 + int(rng.integers(-6, 7))
    cy = raw_size // 2 + int(rng.integers(-6, 7))
    rx, ry = int(raw_size * 0.28), int(raw_size * 0.34)
    cv2.ellipse(img, (cx, cy), (rx, ry), 0.0, 0, 360, 120.0, -1)
    # Dark lateral ventricles
    cv2.ellipse(img, (cx - 14, cy - 8), (8, 18), -12.0, 0, 360, 50.0, -1)
    cv2.ellipse(img, (cx + 14, cy - 8), (8, 18), 12.0, 0, 360, 50.0, -1)

    if kind == "glioma":
        tx, ty = cx - int(rx * 0.45), cy - int(ry * 0.35)
        cv2.circle(img, (tx, ty), 22, 240.0, -1)
        cv2.circle(img, (tx, ty), 7, 150.0, -1)  # necrotic core cue
    elif kind == "meningioma":
        tx, ty = cx + int(rx * 0.60), cy - int(ry * 0.08)
        cv2.ellipse(img, (tx, ty), (18, 24), 10.0, 0, 360, 240.0, -1)
    elif kind == "pituitary":
        tx, ty = cx, cy + int(ry * 0.52)
        cv2.circle(img, (tx, ty), 16, 245.0, -1)
    elif kind == "subtle":
        # Subtle low-contrast lesion for realistic Stage 1 sensitivity threshold search
        cv2.circle(img, (cx, cy + int(ry * 0.52)), 11, 170.0, -1)
    elif kind == "ambiguous":
        # Balanced glioma/meningioma morph so Stage 1 fires positive while Stage 2 abstains (< 0.70)
        img_g = make_synthetic_mri("glioma", seed_idx, raw_size=raw_size)
        img_m = make_synthetic_mri("meningioma", seed_idx, raw_size=raw_size)
        return np.maximum(0.515 * img_g, 0.485 * img_m).astype(np.float32)
    elif kind != "no_tumor":
        raise ValueError(f"Unknown synthetic slice kind: {kind!r}")

    brain_mask = img > 0
    noise = rng.normal(0.0, 7.0, size=(raw_size, raw_size)).astype(np.float32)
    img[brain_mask] = np.clip(img[brain_mask] + noise[brain_mask], 15.0, 255.0)
    return img


# Build lightweight Stage 1 (binary) and Stage 2 (3-class) synthetic train/val datasets
helper_pre = CascadePredictor(
    stage1_model=SimpleCNN(num_classes=2),
    stage2_model=SimpleCNN(num_classes=3),
    image_size=224,
    device="cpu",
)

def build_tensor_split(specs: list[tuple[str, int]], start_seed: int) -> tuple[torch.Tensor, torch.Tensor]:
    xs, ys = [], []
    for idx, (kind, label) in enumerate(specs):
        t, _ = helper_pre.preprocess_array(make_synthetic_mri(kind, start_seed + idx))
        xs.append(t)
        ys.append(label)
    return torch.cat(xs, dim=0), torch.tensor(ys, dtype=torch.long)

s1_train_specs = [("no_tumor", 0)] * 18 + [("glioma", 1)] * 6 + [("meningioma", 1)] * 6 + [("pituitary", 1)] * 6
s1_val_specs = [("no_tumor", 0)] * 8 + [("glioma", 1)] * 3 + [("meningioma", 1)] * 3 + [("pituitary", 1)] * 2 + [("subtle", 1)] * 1
s2_train_specs = [("glioma", 0)] * 12 + [("meningioma", 1)] * 12 + [("pituitary", 2)] * 12
s2_val_specs = [("glioma", 0)] * 5 + [("meningioma", 1)] * 5 + [("pituitary", 2)] * 5

x1_tr, y1_tr = build_tensor_split(s1_train_specs, start_seed=10)
x1_va, y1_va = build_tensor_split(s1_val_specs, start_seed=100)
x2_tr, y2_tr = build_tensor_split(s2_train_specs, start_seed=200)
x2_va, y2_va = build_tensor_split(s2_val_specs, start_seed=300)

def quick_train(model: nn.Module, x_tr: torch.Tensor, y_tr: torch.Tensor, epochs: int = 35, lr: float = 2e-3, label_smoothing: float = 0.0) -> nn.Module:
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=8, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(xb.to(device)), yb.to(device))
            loss.backward()
            opt.step()
    model.eval()
    return model

stage1_model = quick_train(SimpleCNN(num_classes=2, dropout=0.1), x1_tr, y1_tr, epochs=35, lr=2e-3, label_smoothing=0.1)
stage2_model = quick_train(
    TimmClassifier(
        backbone_name="efficientnet_b0",
        num_classes=3,
        pretrained=False,
        dropout=0.1,
    ),
    x2_tr,
    y2_tr,
    epochs=25,
    lr=3e-3,
    label_smoothing=0.0,
)

# Post-hoc Stage 1 sensitivity threshold & Stage 2 temperature scaling
with torch.no_grad():
    s1_val_logits = stage1_model(x1_va.to(device)).cpu()
    s1_val_probs = torch.softmax(s1_val_logits, dim=-1)[:, 1].numpy()
    s2_val_logits = stage2_model(x2_va.to(device)).cpu().numpy()

thr_res = find_sensitivity_threshold(s1_val_probs, y1_va.numpy(), target_sensitivity=0.98)
temp_scaler = TemperatureScaler()
fitted_temp = temp_scaler.fit(s2_val_logits, y2_va.numpy())

print(f"Stage 1 calibrated threshold : {thr_res.threshold:.4f} (val sens={thr_res.sensitivity:.3f}, spec={thr_res.specificity:.3f})")
print(f"Stage 2 fitted temperature   : {fitted_temp:.4f}")

## 3. Three-Way Clinical Routing via `CascadePredictor`

We assemble `CascadePredictor` with our calibrated `stage1_threshold`, `stage2_temperature`, and `stage2_uncertainty_threshold = 0.70`, then test the three clinical routing branches:
1. **Healthy Clearance (`no_tumor`)**: Stage 1 exits early (`stage1_positive=False`, `stage2_probs=None`).
2. **Confident Tumor Subtyping (`glioma` / `meningioma` / `pituitary`)**: Stage 1 fires positive and $\max_c \hat{p}_c \ge 0.70$.
3. **Confidence Abstention (`ABSTAIN_LABEL`)**: Stage 1 fires positive on an ambiguous lesion, $\max_c \hat{p}_c < 0.70$ triggers `abstained=True`, while `raw_4way_label` preserves the argmax subtype for audit metrics.

In [ ]:
predictor = CascadePredictor(
    stage1_model=stage1_model,
    stage2_model=stage2_model,
    stage1_threshold=thr_res.threshold,
    stage2_temperature=fitted_temp,
    stage2_uncertainty_threshold=0.70,
    image_size=224,
    device=device,
)

img_healthy = make_synthetic_mri("no_tumor", seed_idx=701)
img_confident = make_synthetic_mri("meningioma", seed_idx=702)
img_ambiguous = make_synthetic_mri("ambiguous", seed_idx=703)

res_healthy: CascadeResult = predictor.predict_image(img_healthy, return_gradcam=True, gradcam_on_negative=False)
res_confident: CascadeResult = predictor.predict_image(img_confident, return_gradcam=True)
res_ambig: CascadeResult = predictor.predict_image(img_ambiguous, return_gradcam=True)

if not res_ambig.abstained:
    img_g = make_synthetic_mri("glioma", seed_idx=703)
    img_m = make_synthetic_mri("meningioma", seed_idx=703)
    for alpha in np.linspace(0.40, 0.55, 31):
        cand_img = np.maximum((1.0 - alpha) * img_g, alpha * img_m).astype(np.float32)
        cand = predictor.predict_image(cand_img, return_gradcam=True)
        if cand.stage1_positive and cand.abstained:
            res_ambig = cand
            break

assert not res_healthy.stage1_positive and res_healthy.predicted_label == "no_tumor"
assert res_healthy.stage2_probs is None and not res_healthy.abstained
assert res_confident.stage1_positive and not res_confident.abstained
assert res_confident.predicted_label == "meningioma"
assert res_ambig.stage1_positive and res_ambig.abstained
assert res_ambig.predicted_label == ABSTAIN_LABEL
assert res_ambig.raw_4way_label in {"glioma", "meningioma", "pituitary"}

for case_name, r in [
    ("(a) Healthy Clearance", res_healthy),
    ("(b) Confident Subtype", res_confident),
    ("(c) Ambiguous Escalation", res_ambig),
]:
    print(f"=== {case_name} ===")
    print(json.dumps(r.to_dict(include_arrays=False), indent=2))

## 4. Grad-CAM Overlay & Stage 2 Probability Visualization

For every Stage 1-positive detection (`res_confident` and `res_ambig`), `return_gradcam=True` populates `preprocessed_image`, `gradcam_heatmap`, and `gradcam_overlay` (`gradcam_stage == "stage2"`). Below we plot `[Preprocessed Crop | Grad-CAM Overlay | Stage 2 Calibrated Probabilities]` side by side.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8.2))
cases = [
    ("Confident Detection", res_confident),
    ("Ambiguous Tumor (Abstained)", res_ambig),
]

for row_idx, (title_prefix, res) in enumerate(cases):
    assert res.preprocessed_image is not None
    assert res.gradcam_overlay is not None
    assert res.stage2_probs is not None

    ax_crop, ax_cam, ax_bar = axes[row_idx]
    ax_crop.imshow(res.preprocessed_image)
    ax_crop.set_title(f"{title_prefix}\nPreprocessed 224x224 Crop")
    ax_crop.axis("off")

    ax_cam.imshow(res.gradcam_overlay)
    ax_cam.set_title(f"Grad-CAM Overlay ({res.gradcam_stage})\nraw_4way={res.raw_4way_label}")
    ax_cam.axis("off")

    labels = list(res.stage2_probs.keys())
    probs = [res.stage2_probs[k] for k in labels]
    colors = ["#2b8a3e" if not res.abstained else "#e67700" for _ in labels]
    bars = ax_bar.bar(labels, probs, color=colors, alpha=0.88)
    ax_bar.axhline(
        res.stage2_uncertainty_threshold,
        color="#c92a2a",
        linestyle="--",
        linewidth=1.5,
        label=f"tau_abstain = {res.stage2_uncertainty_threshold:.2f}",
    )
    ax_bar.set_ylim(0.0, 1.08)
    ax_bar.set_ylabel("Calibrated Probability")
    status_tag = "ABSTAINED" if res.abstained else f"PRED: {res.predicted_label}"
    ax_bar.set_title(f"{status_tag}\nmax_p={res.stage2_confidence:.3f}, T={res.stage2_temperature:.2f}")
    ax_bar.legend(loc="upper right", fontsize=8.5)
    ax_bar.grid(axis="y", alpha=0.25)
    for bar, p_val in zip(bars, probs):
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2.0,
            p_val + 0.025,
            f"{p_val:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()
plt.close(fig)

## 5. Testing `CascadePredictor.from_configs()` & `calibration.json` Overrides

We persist our trained `SimpleCNN` and `TimmClassifier("efficientnet_b0")` weights plus `calibration.json` inside a temporary project layout (`configs/` + `checkpoints/stage*/`) and verify that `CascadePredictor.from_configs()` automatically overrides the fallback defaults in `cascade.yaml`.

In [ ]:
proj_dir = scratch_dir / "demo_project"
cfg_dir = proj_dir / "configs"
ckpt1_dir = proj_dir / "checkpoints" / "stage1"
ckpt2_dir = proj_dir / "checkpoints" / "stage2"
for d in (cfg_dir, ckpt1_dir, ckpt2_dir):
    d.mkdir(parents=True, exist_ok=True)

torch.save({"model_state_dict": stage1_model.state_dict()}, ckpt1_dir / "best_model.pt")
torch.save({"model_state_dict": stage2_model.state_dict()}, ckpt2_dir / "best_model.pt")

# Write calibrated parameters distinct from cascade.yaml defaults (0.35, 1.0, 0.70)
saved_thr = float(thr_res.threshold)
saved_temp = float(fitted_temp)
saved_unc = 0.72

(ckpt1_dir / "calibration.json").write_text(
    json.dumps({"stage": "stage1", "threshold": saved_thr}, indent=2),
    encoding="utf-8",
)
(ckpt2_dir / "calibration.json").write_text(
    json.dumps(
        {"stage": "stage2", "temperature": saved_temp, "uncertainty_threshold": saved_unc},
        indent=2,
    ),
    encoding="utf-8",
)

(cfg_dir / "stage1.yaml").write_text(
    yaml.safe_dump({
        "stage": "stage1",
        "data": {"image_size": 224, "use_clahe": False, "crop_margin_ratio": 0.05},
        "model": {"arch": "simple_cnn", "backbone": "simple_cnn", "pretrained": True, "num_classes": 2, "dropout": 0.1},
    }),
    encoding="utf-8",
)
(cfg_dir / "stage2.yaml").write_text(
    yaml.safe_dump({
        "stage": "stage2",
        "data": {"image_size": 224, "use_clahe": False, "crop_margin_ratio": 0.05},
        "model": {
            "arch": "cnn",
            "backbone": "efficientnet_b0",
            "pretrained": True,
            "num_classes": 3,
            "dropout": 0.1,
        },
    }),
    encoding="utf-8",
)
cas_yaml_path = cfg_dir / "cascade.yaml"
cas_yaml_path.write_text(
    yaml.safe_dump({
        "stage1_config": "configs/stage1.yaml",
        "stage2_config": "configs/stage2.yaml",
        "stage1_checkpoint": "checkpoints/stage1/best_model.pt",
        "stage2_checkpoint": "checkpoints/stage2/best_model.pt",
        "image_size": 224,
        "stage1_threshold": 0.35,
        "stage2_temperature": 1.0,
        "stage2_uncertainty_threshold": 0.70,
    }),
    encoding="utf-8",
)

loaded_predictor = CascadePredictor.from_configs(cas_yaml_path, device=device, load_weights=True)

assert abs(loaded_predictor.stage1_threshold - saved_thr) < 1e-6
assert abs(loaded_predictor.stage2_temperature - saved_temp) < 1e-6
assert abs(loaded_predictor.stage2_uncertainty_threshold - saved_unc) < 1e-6

reloaded_res = loaded_predictor.predict_image(img_confident, return_gradcam=False)
assert reloaded_res.predicted_label == res_confident.predicted_label

print("CascadePredictor.from_configs() calibration overrides verified:")
print(f"  stage1_threshold           : 0.3500 (YAML) -> {loaded_predictor.stage1_threshold:.4f} (calibration.json)")
print(f"  stage2_temperature         : 1.0000 (YAML) -> {loaded_predictor.stage2_temperature:.4f} (calibration.json)")
print(f"  stage2_uncertainty_threshold: 0.7000 (YAML) -> {loaded_predictor.stage2_uncertainty_threshold:.4f} (calibration.json)")
print(f"  Reloaded prediction check  : {reloaded_res.predicted_label} (P_tumor={reloaded_res.stage1_tumor_prob:.4f})")

## 6. Batched Cascade Inference (`predict_batch_tensors`) & `4x5` Clinical Routing Matrix

Finally, we evaluate `predictor.predict_batch_tensors` on a synthetic test cohort containing all 4 classes (`no_tumor`, `glioma`, `meningioma`, `pituitary`, including low-contrast borderline lesions), compute `compute_cascade_metrics`, and render the `4x5` clinical routing matrix with `plot_confusion_matrix`.

In [ ]:
test_specs: list[tuple[str, str]] = (
    [("no_tumor", "no_tumor")] * 10
    + [("glioma", "glioma")] * 8
    + [("ambiguous", "glioma")] * 2
    + [("meningioma", "meningioma")] * 8
    + [("ambiguous", "meningioma")] * 2
    + [("pituitary", "pituitary")] * 10
)

batch_tensors = []
true_labels: list[str] = []
for idx, (synth_kind, true_cls) in enumerate(test_specs):
    t, _ = predictor.preprocess_array(make_synthetic_mri(synth_kind, seed_idx=900 + idx))
    batch_tensors.append(t)
    true_labels.append(true_cls)

batch_tensor = torch.cat(batch_tensors, dim=0)
batch_results = predictor.predict_batch_tensors(batch_tensor)

pred_labels = [r.predicted_label for r in batch_results]
raw_4way_preds = [r.raw_4way_label for r in batch_results]

cascade_metrics = compute_cascade_metrics(
    true_labels=true_labels,
    pred_labels=pred_labels,
    raw_4way_preds=raw_4way_preds,
)

print(f"Evaluated cohort size      : {cascade_metrics.n_samples}")
print(f"cascade_tumor_sensitivity  : {cascade_metrics.cascade_tumor_sensitivity:.4f}")
print(f"cascade_healthy_specificity: {cascade_metrics.cascade_healthy_specificity:.4f}")
print(f"abstention_rate            : {cascade_metrics.abstention_rate:.4f}")
print(f"overall_4way_accuracy      : {cascade_metrics.overall_4way_accuracy:.4f}")
print(f"selective_4way_accuracy    : {cascade_metrics.selective_4way_accuracy}")
print(f"Routing columns            : {cascade_metrics.routing_columns}")
print(f"4x5 Routing matrix         : {cascade_metrics.routing_matrix_4x5}")

routing_plot_path = plot_confusion_matrix(
    cascade_metrics.routing_matrix_4x5,
    class_names=cascade_metrics.routing_columns,
    output_path=scratch_dir / "cascade_routing_matrix_4x5.png",
    title="Cascade 4x5 Clinical Routing Matrix (Synthetic Batch)",
)
show_saved_plot(routing_plot_path, figsize=(7.2, 5.0))

scratch_ctx.cleanup()
print("Temporary scratch directory cleaned up cleanly.")